# Protein Data Handling: FASTA and PDB Files

This notebook covers loading and processing common protein file formats.

**Learning Objectives:**
- Parse FASTA files for sequence data
- Load PDB files and extract structure information
- Visualize protein structures
- Prepare data for downstream analysis

In [ ]:
# Install dependencies (run on Colab)
# !pip install biopython biotite py3Dmol -q

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt

# Check imports
try:
    from Bio import SeqIO
    print("Biopython: OK")
except ImportError:
    print("Biopython not installed. Run: pip install biopython")

try:
    import biotite.structure as struc
    import biotite.structure.io.pdb as pdb
    from biotite.database import rcsb
    print("Biotite: OK")
except ImportError:
    print("Biotite not installed. Run: pip install biotite")

try:
    import py3Dmol
    print("py3Dmol: OK")
except ImportError:
    print("py3Dmol not installed. Run: pip install py3Dmol")

## 1. FASTA File Format

FASTA is the simplest format for storing biological sequences.

```
>header_line|additional_info
SEQUENCESEQUENCESEQUENCE
MORESEQUENCEMORESEQUENCE
```

In [ ]:
# Create a sample FASTA file
sample_fasta = """>sp|P0A6Y8|DNAK_ECOLI Chaperone protein DnaK OS=Escherichia coli
MGKIIGIDLGTTNSCVAIMDGTTPRVLENAEGDRTTPSIIAYTQDGETLVGQPAKRQAVT
NPQNTLFAIKRLIGRRFQDEEVQRDVSIMPFKIIAADNGDAWVEVKGQKMAPPQISAEVL
KKMKKTAEDYLGEPVTEAVITVPAYFNDAQRQATKDAGRIAGLEVKRIINEPTAAALAYGLD
>sp|P0AFG8|ODO1_ECOLI 2-oxoglutarate dehydrogenase E1 component
MSERFPNDVDPIETRDWLQAIESVIREEGVERAQYLIDQLLAEARKGGVNVAAGTGISNY
INTIPVEEQPEYPGNLELERRIRSAIRWNAIMTVLRASKKDLELGGHMASFQSSATIYDV
>sp|P62942|FKB1A_HUMAN Peptidyl-prolyl cis-trans isomerase FKBP1A
MGVQVETISPGDGRTFPKRGQTCVVHYTGMLEDGKKFDSSRDRNKPFKFMLGKQEVIRGW
EEGVAQMSVGQRAKLTISPDYAYGATGHPGIIPPHATLVFDVELLKLE
"""

# Save to file
with open('sample_proteins.fasta', 'w') as f:
    f.write(sample_fasta)

print("Sample FASTA file created.")

In [ ]:
# Parse FASTA with Biopython
from Bio import SeqIO

sequences = {}
for record in SeqIO.parse('sample_proteins.fasta', 'fasta'):
    sequences[record.id] = {
        'sequence': str(record.seq),
        'description': record.description,
        'length': len(record.seq)
    }

print(f"Loaded {len(sequences)} sequences:\n")
for seq_id, info in sequences.items():
    print(f"{seq_id}")
    print(f"  Length: {info['length']}")
    print(f"  First 50 aa: {info['sequence'][:50]}...")
    print()

In [ ]:
# Analyze sequences
from collections import Counter

def analyze_sequence(sequence):
    """Compute basic sequence statistics."""
    counts = Counter(sequence)
    length = len(sequence)
    
    # Amino acid categories
    hydrophobic = sum(counts.get(aa, 0) for aa in 'AILMFWV')
    charged = sum(counts.get(aa, 0) for aa in 'DEKR')
    polar = sum(counts.get(aa, 0) for aa in 'NQSTY')
    
    return {
        'length': length,
        'hydrophobic_frac': hydrophobic / length,
        'charged_frac': charged / length,
        'polar_frac': polar / length,
    }

for seq_id, info in sequences.items():
    stats = analyze_sequence(info['sequence'])
    print(f"{seq_id}:")
    print(f"  Hydrophobic: {stats['hydrophobic_frac']:.1%}")
    print(f"  Charged: {stats['charged_frac']:.1%}")
    print(f"  Polar: {stats['polar_frac']:.1%}")
    print()

## 2. PDB File Format

PDB files contain 3D atomic coordinates of protein structures.

In [ ]:
# Download a PDB file from RCSB
from biotite.database import rcsb

# Download ubiquitin (1UBQ) - a small well-studied protein
pdb_path = rcsb.fetch('1UBQ', 'pdb', target_path='.')
print(f"Downloaded: {pdb_path}")

In [ ]:
# Load and examine the structure
import biotite.structure as struc
import biotite.structure.io.pdb as pdb

pdb_file = pdb.PDBFile.read(pdb_path)
structure = pdb_file.get_structure(model=1)  # Get first model

print(f"Total atoms: {len(structure)}")
print(f"Chains: {np.unique(structure.chain_id)}")
print(f"Atom types: {np.unique(structure.atom_name)[:10]}...")  # First 10 atom types

In [ ]:
# Filter to protein atoms only
protein = structure[struc.filter_amino_acids(structure)]
print(f"Protein atoms: {len(protein)}")

# Get residue information
residue_ids = struc.get_residues(protein)[0]
print(f"Number of residues: {len(residue_ids)}")

In [ ]:
# Extract CA (alpha carbon) coordinates
ca_mask = protein.atom_name == 'CA'
ca_atoms = protein[ca_mask]
ca_coords = ca_atoms.coord

print(f"CA atoms: {len(ca_atoms)}")
print(f"CA coordinates shape: {ca_coords.shape}")
print(f"\nFirst 5 CA positions:")
print(ca_coords[:5])

In [ ]:
# Extract sequence from structure
AA_3TO1 = {
    'ALA': 'A', 'CYS': 'C', 'ASP': 'D', 'GLU': 'E', 'PHE': 'F',
    'GLY': 'G', 'HIS': 'H', 'ILE': 'I', 'LYS': 'K', 'LEU': 'L',
    'MET': 'M', 'ASN': 'N', 'PRO': 'P', 'GLN': 'Q', 'ARG': 'R',
    'SER': 'S', 'THR': 'T', 'VAL': 'V', 'TRP': 'W', 'TYR': 'Y'
}

# Get residue names for CA atoms
residue_names = ca_atoms.res_name
sequence = ''.join(AA_3TO1.get(name, 'X') for name in residue_names)

print(f"Sequence ({len(sequence)} residues):")
print(sequence)

## 3. Backbone Coordinates and Dihedral Angles

In [ ]:
def extract_backbone(structure):
    """
    Extract backbone atom coordinates (N, CA, C, O) for each residue.
    
    Returns:
        coords: (N_residues, 4, 3) array
        sequence: amino acid sequence string
    """
    protein = structure[struc.filter_amino_acids(structure)]
    residue_ids = struc.get_residues(protein)[0]
    
    backbone_atoms = ['N', 'CA', 'C', 'O']
    n_residues = len(residue_ids)
    coords = np.zeros((n_residues, 4, 3), dtype=np.float32)
    sequence = []
    
    for i, res_id in enumerate(residue_ids):
        res_mask = protein.res_id == res_id
        res_atoms = protein[res_mask]
        
        # Get sequence
        res_name = res_atoms.res_name[0]
        sequence.append(AA_3TO1.get(res_name, 'X'))
        
        # Extract backbone coordinates
        for j, atom_name in enumerate(backbone_atoms):
            atom_mask = res_atoms.atom_name == atom_name
            if np.any(atom_mask):
                coords[i, j] = res_atoms.coord[atom_mask][0]
    
    return coords, ''.join(sequence)

backbone_coords, seq = extract_backbone(structure)
print(f"Backbone coordinates shape: {backbone_coords.shape}")
print(f"(residues, atoms[N,CA,C,O], xyz)")

In [ ]:
# Compute dihedral angles (phi, psi)
def compute_dihedral(p1, p2, p3, p4):
    """Compute dihedral angle between four points."""
    b1 = p2 - p1
    b2 = p3 - p2
    b3 = p4 - p3
    
    n1 = np.cross(b1, b2)
    n2 = np.cross(b2, b3)
    
    n1 = n1 / (np.linalg.norm(n1) + 1e-10)
    n2 = n2 / (np.linalg.norm(n2) + 1e-10)
    
    m1 = np.cross(n1, b2 / (np.linalg.norm(b2) + 1e-10))
    
    x = np.dot(n1, n2)
    y = np.dot(m1, n2)
    
    return np.arctan2(y, x)

# Compute phi and psi for each residue
n_res = len(backbone_coords)
phi = np.zeros(n_res)
psi = np.zeros(n_res)

for i in range(n_res):
    # phi: C(i-1) - N(i) - CA(i) - C(i)
    if i > 0:
        phi[i] = compute_dihedral(
            backbone_coords[i-1, 2],  # C(i-1)
            backbone_coords[i, 0],    # N(i)
            backbone_coords[i, 1],    # CA(i)
            backbone_coords[i, 2]     # C(i)
        )
    
    # psi: N(i) - CA(i) - C(i) - N(i+1)
    if i < n_res - 1:
        psi[i] = compute_dihedral(
            backbone_coords[i, 0],    # N(i)
            backbone_coords[i, 1],    # CA(i)
            backbone_coords[i, 2],    # C(i)
            backbone_coords[i+1, 0]   # N(i+1)
        )

# Convert to degrees
phi_deg = np.degrees(phi)
psi_deg = np.degrees(psi)

print(f"Phi range: [{phi_deg.min():.1f}, {phi_deg.max():.1f}] degrees")
print(f"Psi range: [{psi_deg.min():.1f}, {psi_deg.max():.1f}] degrees")

In [ ]:
# Ramachandran plot
plt.figure(figsize=(8, 8))
plt.scatter(phi_deg[1:-1], psi_deg[1:-1], alpha=0.7, s=50)
plt.xlabel('Phi (degrees)')
plt.ylabel('Psi (degrees)')
plt.title('Ramachandran Plot - Ubiquitin (1UBQ)')
plt.xlim(-180, 180)
plt.ylim(-180, 180)
plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
plt.grid(True, alpha=0.3)
plt.show()

## 4. Distance and Contact Maps

In [ ]:
# Compute distance matrix from CA coordinates
def compute_distance_matrix(coords):
    diff = coords[:, np.newaxis, :] - coords[np.newaxis, :, :]
    return np.sqrt(np.sum(diff ** 2, axis=-1))

dist_matrix = compute_distance_matrix(ca_coords)
print(f"Distance matrix shape: {dist_matrix.shape}")

In [ ]:
# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Distance matrix
im1 = axes[0].imshow(dist_matrix, cmap='viridis', origin='lower')
axes[0].set_xlabel('Residue Index')
axes[0].set_ylabel('Residue Index')
axes[0].set_title('CA-CA Distance Matrix')
plt.colorbar(im1, ax=axes[0], label='Distance (Å)')

# Contact map (8Å threshold)
contact_map = dist_matrix < 8.0
axes[1].imshow(contact_map, cmap='Blues', origin='lower')
axes[1].set_xlabel('Residue Index')
axes[1].set_ylabel('Residue Index')
axes[1].set_title('Contact Map (8Å threshold)')

plt.tight_layout()
plt.show()

## 5. 3D Visualization with py3Dmol

In [ ]:
import py3Dmol

# Read PDB file content
with open(pdb_path) as f:
    pdb_string = f.read()

# Create viewer
viewer = py3Dmol.view(width=600, height=400)
viewer.addModel(pdb_string, 'pdb')

# Set style - cartoon colored by secondary structure
viewer.setStyle({'cartoon': {'color': 'spectrum'}})
viewer.zoomTo()
viewer.show()

In [ ]:
# Different visualization styles
viewer2 = py3Dmol.view(width=600, height=400)
viewer2.addModel(pdb_string, 'pdb')

# Show backbone as sticks, sidechains as lines
viewer2.setStyle({'atom': 'CA'}, {'sphere': {'radius': 0.5, 'color': 'red'}})
viewer2.setStyle({'atom': ['N', 'C', 'O']}, {'stick': {'radius': 0.2}})
viewer2.setStyle({'and': [{'atom': ['N', 'C', 'O', 'CA'], 'invert': True}]}, 
                 {'line': {'color': 'gray'}})
viewer2.zoomTo()
viewer2.show()

## 6. Combining Sequence and Structure Data

In [ ]:
import pandas as pd

# Create a comprehensive residue-level dataframe
residue_data = []

for i in range(len(seq)):
    residue_data.append({
        'residue_idx': i,
        'residue_id': residue_ids[i],
        'amino_acid': seq[i],
        'x': ca_coords[i, 0],
        'y': ca_coords[i, 1],
        'z': ca_coords[i, 2],
        'phi': phi_deg[i] if i > 0 else np.nan,
        'psi': psi_deg[i] if i < len(seq)-1 else np.nan,
        'n_contacts': (contact_map[i].sum() - 1),  # Exclude self
    })

residue_df = pd.DataFrame(residue_data)
residue_df.head(10)

In [ ]:
# Analyze by amino acid type
aa_stats = residue_df.groupby('amino_acid').agg({
    'residue_idx': 'count',
    'n_contacts': 'mean',
}).rename(columns={'residue_idx': 'count'})

aa_stats = aa_stats.sort_values('count', ascending=False)
print("Amino acid statistics for ubiquitin:")
aa_stats

In [ ]:
# Visualize contacts along sequence
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# Number of contacts per residue
axes[0].bar(residue_df['residue_idx'], residue_df['n_contacts'], 
           color='steelblue', alpha=0.7)
axes[0].set_ylabel('Number of Contacts')
axes[0].set_title('Contact Density Along Sequence')

# Color by amino acid hydrophobicity
hydrophobic = 'AILMFWV'
colors = ['red' if aa in hydrophobic else 'blue' for aa in seq]
axes[1].bar(residue_df['residue_idx'], residue_df['n_contacts'], 
           color=colors, alpha=0.7)
axes[1].set_xlabel('Residue Index')
axes[1].set_ylabel('Number of Contacts')
axes[1].set_title('Hydrophobic (red) vs Polar (blue) Residues')

plt.tight_layout()
plt.show()

## Summary

In this notebook, we covered:

1. **FASTA format** - parsing sequences with Biopython
2. **PDB format** - loading structures with Biotite
3. **Backbone coordinates** - extracting N, CA, C, O positions
4. **Dihedral angles** - computing phi/psi for Ramachandran plots
5. **Distance/Contact maps** - structural analysis
6. **3D visualization** - interactive viewing with py3Dmol
7. **Data integration** - combining sequence and structure in DataFrames

These skills are essential for preparing protein data for machine learning.

In [ ]:
# Cleanup
import os
if os.path.exists('sample_proteins.fasta'):
    os.remove('sample_proteins.fasta')
print("Cleanup complete.")